# Resonator With Indium Bumps - Route C - No Inset

This notebook builds one route XAO file and prints final physical groups.

This fixture is the fast Indium-bump contact case; `sim_flip_chip_distance` remains the large sample case.

Inset breakpoints: `0, 50 nm, 100 nm, 200 nm, 500 nm, 1 um`.
Sidewall inset bands that would degenerate are skipped by the planner.


In [ ]:
from pathlib import Path
import json

import gdstk
from semantic_geometry_builder import (
    SemanticGeometryBuilder,
    build_gds_stack_geometry_input,
)

GEOMETRY_ID = 'resonator_with_indium_bumps'
ROUTE = 'C'
MODE = 'no_inset'
BUILD_SUPPORTED = True
INSET_BREAKPOINTS_UM = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
TOP_CELL_NAME = 'resonator_with_indium_bumps_RL4000_RM6_BFP360_IBS20_IBG_ab09c4a0'

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent

ASSET_DIR = ROOT / "tutorials" / "assets"
RUN_FOLDER = ROOT / "tutorials" / "runs" / MODE / GEOMETRY_ID / f"route_{ROUTE.lower()}"
GDS_FILE = ASSET_DIR / f"{GEOMETRY_ID}.gds"
STACK_FILE = ASSET_DIR / f"{GEOMETRY_ID}.stack.json"
METADATA_OVERRIDE = {"inset_routes": []}

build_input = build_gds_stack_geometry_input(
    gds_file=GDS_FILE,
    stack_file=STACK_FILE,
    top_cell_name=TOP_CELL_NAME,
    metadata=METADATA_OVERRIDE,
)
print("polygons:", len(build_input.polygons))
print("entities:", len(build_input.entities))
print("inset routes:", build_input.metadata.get("inset_routes"))


In [ ]:
stack = json.loads(STACK_FILE.read_text())
if TOP_CELL_NAME is not None:
    cell = next(c for c in gdstk.read_gds(str(GDS_FILE)).cells if c.name == TOP_CELL_NAME)
    gds_layers = {(int(p.layer), int(p.datatype)) for p in cell.get_polygons(apply_repetitions=True)}
    layer_records = {(int(r["layer"]), int(r["datatype"])) for r in stack.get("layers", [])}
    ignored = {(int(r["layer"]), int(r["datatype"])) for r in stack["metadata"].get("ignored_layout_layers", [])}
    deferred = {(int(r["layer"]), int(r["datatype"])) for r in stack["metadata"].get("deferred_high_count_layers", [])}
    unclassified = sorted(gds_layers - layer_records - ignored - deferred)
    print("unclassified layers:", unclassified)
    print("ignored layers:", sorted(ignored))
    assert not unclassified
    assert (40, 1) in ignored, "Under-bump layer must stay ignored"


In [ ]:
if not BUILD_SUPPORTED:
    print("Builder intentionally not run for this fixture yet.")
else:
    groups = SemanticGeometryBuilder().build(
        build_input,
        route=ROUTE,
        run_folder=RUN_FOLDER,
    )
    xao_path = RUN_FOLDER / "geometry" / f"semantic_geometry_route_{ROUTE.lower()}.xao"
    print("XAO:", xao_path)
    print("groups:", len(groups))
    for group in groups:
        print(group.dimension, group.physical_name, len(group.entity_tags))
    assert xao_path.is_file()
    assert not list(RUN_FOLDER.rglob("*.msh"))
